# Envanter hakem koşusu — v4 + havuz kapısı (V4EC)

`needs_review_subset_reversed.csv` (143.039 sorgu) → `gemma4:e4b` hakem.

## Önceki koşudan farkı

| | eski | bu koşu |
|---|---|---|
| prompt | v1 | **v4** — şemanın zaten zorladığı 2 kural bloğu çıkarıldı |
| havuz kapısı | yok | **chosen** — seçilen kayıt sorgunun kimlik kelimesini taşımıyorsa `auto_match`→`review` |
| ölçüm | — | 125 sorguda v4: 27 iyi/8 kötü · kapı: 30 satırlık kontrolde 2 hatanın 2'sini yakaladı |

Eski koşunun ~19.400 satırı **kullanılmayacak** (v1 + kapısız üretildi).

## Drive yapısı (hazır olmalı)
```
institution_resolver_v3/
  data_processed/   parent_canonical.jsonl · subunit_canonical.jsonl · embeddings.npz
  jobs/             needs_review_subset_reversed.csv
  ollama_models/    gemma4:e4b  (manifests/ + blobs/, ~8,9 GB ağırlık dosyası dahil)
  output/           koşu çıktısı buraya yedeklenir
```

`embeddings.npz` mevcut olduğu için indeksleme yeniden **encode etmez** (~20 dk kazanç).

## Çalıştırma
**Save Version → Save & Run All (Commit).** İnteraktif oturum tarayıcı kopunca
kapanıyor; geçen sefer ilerleme o yüzden kaybolmuştu. Bu notebook başsız koşacak
şekilde yazıldı ve her `CHUNK` satırda Drive'a yedekliyor.


## 1) GPU


In [ ]:
!nvidia-smi
import subprocess
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or '(YOK)')
assert gpu, 'GPU yok - Runtime -> Change runtime type -> GPU'


## 2) Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT      = '/content/drive/MyDrive/institution_resolver_v3'
PROCESSED = f'{ROOT}/data_processed'
JOBS      = f'{ROOT}/jobs'
OLLAMA    = f'{ROOT}/ollama_models'
OUTPUT    = f'{ROOT}/output'
os.makedirs(OUTPUT, exist_ok=True)

GIRDI = f'{JOBS}/needs_review_subset_reversed.csv'
for p in (PROCESSED, OLLAMA):
    assert os.path.isdir(p) and os.listdir(p), f'Drive eksik: {p}'
assert os.path.isfile(GIRDI), f'girdi yok: {GIRDI}'
assert os.path.isfile(f'{PROCESSED}/embeddings.npz'), 'embeddings.npz yok'

# gemma4:e4b agirlik dosyasi gercekten kopyalanmis mi (8,9 GB) - manifest
# var ama blob eksik olabiliyor, gecen sefer oyle oldu.
import json, glob
man = f'{OLLAMA}/manifests/registry.ollama.ai/library/gemma4/e4b'
assert os.path.isfile(man), f'gemma4:e4b manifesti yok: {man}'
eksik = []
for l in [json.load(open(man))['config']] + json.load(open(man))['layers']:
    b = f"{OLLAMA}/blobs/{l['digest'].replace(':','-')}"
    if not os.path.exists(b):
        eksik.append((l['digest'][:20], l['size']/1e9))
assert not eksik, f'EKSIK BLOB: {eksik} - yerelden ~/.ollama/models/blobs kopyalayin'
print('Drive tamam. Girdi:', GIRDI)


## 3) Kod


In [ ]:
REPO = '/content/institution_resolver_v3'
import os
if not os.path.isdir(REPO):
    !git clone --branch feat/gate-asama1 https://github.com/mcangultekin/institution_resolver_v3.git {REPO}
else:
    !cd {REPO} && git fetch origin && git checkout feat/gate-asama1 && git pull
os.chdir(REPO)
!git log --oneline -1
assert os.path.isfile('src/institution_resolver_v3/retrieve/token_df.py'), 'kapı kodu yok - git pull?'


## 4) Drive sembolik linkleri


In [ ]:
import os, shutil

def _link(ad, hedef):
    link = f'data/{ad}'
    if os.path.islink(link):
        return
    if os.path.isdir(link):
        shutil.rmtree(link)
    os.symlink(hedef, link)

os.makedirs('data', exist_ok=True)
_link('processed', PROCESSED)
_link('jobs', JOBS)
!ls -la data/processed | head -5


## 5) Elasticsearch


In [ ]:
%%bash
set -e
if [ ! -d /content/es ]; then
  wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-8.14.0-linux-x86_64.tar.gz -O /content/es.tar.gz
  mkdir -p /content/es && tar -xzf /content/es.tar.gz -C /content/es --strip-components=1
fi
grep -q '^discovery.type' /content/es/config/elasticsearch.yml || cat >> /content/es/config/elasticsearch.yml <<EOF
discovery.type: single-node
xpack.security.enabled: false
xpack.security.http.ssl.enabled: false
xpack.ml.enabled: false
EOF
cat > /content/es/config/elasticsearch.policy <<'POLEOF'
grant {
  permission java.io.FilePermission "/sys/-", "read";
  permission java.io.FilePermission "/proc/-", "read";
};
POLEOF
sysctl -w vm.max_map_count=262144 || true
id -u esuser &>/dev/null || useradd -m esuser
chown -R esuser:esuser /content/es
pkill -f 'org.elasticsearch.bootstrap.Elasticsearch' 2>/dev/null || true
sleep 1
sudo -u esuser env ES_JAVA_OPTS="-Xms4g -Xmx4g -Djava.security.policy=/content/es/config/elasticsearch.policy" \
  setsid /content/es/bin/elasticsearch < /dev/null > /content/es/es.log 2>&1 &
disown
sleep 3


In [ ]:
import time, requests
for _ in range(90):
    try:
        if requests.get('http://localhost:9200/_cluster/health', timeout=5).status_code == 200:
            print('ES:', requests.get('http://localhost:9200/_cluster/health', timeout=5).json()['status']); break
    except requests.exceptions.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError('ES baslatilamadi - /content/es/es.log')


## 6) Paketler


In [ ]:
import os; os.chdir(REPO)
!pip uninstall -y torchaudio torchvision 2>/dev/null || true
!pip install -q --force-reinstall -e '.[dev,embed,llm,api]'


## 7) Ollama + gemma4:e4b

Model Drive'dan yerel SSD'ye kopyalanır — Drive üzerinden okumak inference'ı yavaşlatır.


In [ ]:
%%bash
set -e
apt-get update -qq && apt-get install -y -qq zstd
if ! command -v ollama &> /dev/null; then
  curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr
fi
ollama --version || true


In [ ]:
import os, shutil, subprocess, time, requests

YEREL_MODEL = '/content/ollama_local'
if not os.path.isdir(YEREL_MODEL):
    print('Model Drive -> SSD kopyalaniyor (~9 GB, birkac dakika)...')
    shutil.copytree(OLLAMA, YEREL_MODEL)
    print('kopyalandi')

os.environ['OLLAMA_MODELS']       = YEREL_MODEL
os.environ['OLLAMA_KEEP_ALIVE']   = '-1'
os.environ['OLLAMA_NUM_PARALLEL'] = '4'   # --workers 4 ile uyumlu

def _tags(t):
    return requests.get('http://localhost:11434/api/tags', timeout=t)

ayakta = False
try:
    ayakta = _tags(5).status_code == 200
except requests.exceptions.RequestException:
    pass
if not ayakta:
    subprocess.Popen(['ollama','serve'], stdout=open('/content/ollama.log','w'),
                     stderr=subprocess.STDOUT, env=os.environ)

names = None
for _ in range(40):
    try:
        # timeout=30: model listesi buyuk dizinden taraniyor, 2 sn yetmiyor.
        # RequestException hem ConnectionError hem ReadTimeout'u kapsar.
        r = _tags(30)
        if r.status_code == 200:
            names = [m['name'] for m in r.json().get('models', [])]; break
    except requests.exceptions.RequestException:
        pass
    time.sleep(3)
if names is None:
    print(open('/content/ollama.log').read()[-3000:]); raise RuntimeError('Ollama baslamadi')
print('Ollama:', names)
assert any('gemma4' in n for n in names), f'gemma4:e4b yok: {names}'

r = requests.post('http://localhost:11434/api/generate',
                  json={'model':'gemma4:e4b','prompt':'Merhaba','stream':False}, timeout=600)
print('warm-up:', 'OK' if r.status_code == 200 else r.text[:200])


## 8) İndeksleme

`embeddings.npz` Drive'da olduğu için encode adımı atlanır; yalnız bulk yükleme yapılır.


In [ ]:
import os; os.chdir(REPO)
!python3 -m institution_resolver_v3.cli.main setup-es
!python3 -m institution_resolver_v3.cli.main index --embeddings
import requests
n = requests.get('http://localhost:9200/institutions_v1/_count', timeout=60).json()['count']
print('indekslenen kayit:', n)
assert n == 231291, f'beklenen 231291, gelen {n}'


## 9) Duman testi — v4 ve kapı gerçekten devrede mi

Üç sorgu, ikisi bilinen hata vakası. Beklenen: kapı `australia` ve `bakanligi`
kelimelerinde ateşleyip `auto_match`'i `review`'a indirmeli, `AFAD`'a dokunmamalı.


In [ ]:
import csv, os
os.chdir(REPO); os.makedirs('/content/tmp', exist_ok=True)
with open('/content/tmp/duman.csv','w',newline='',encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['query','normalized_name','rows']); w.writeheader()
    for q in ['University of South Australia','AFAD','T.C. Ticaret Bakanlığı']:
        w.writerow({'query':q,'normalized_name':q.lower(),'rows':'1'})

!python3 -m institution_resolver_v3.cli.main inventory-batch /content/tmp/duman.csv --out /content/tmp/duman_out.csv

csv.field_size_limit(10_000_000)
rows = list(csv.DictReader(open('/content/tmp/duman_out.csv', encoding='utf-8')))
for r in rows:
    print(f"{r['query'][:30]:<32} prompt={r['prompt_variant']:<3} kapi={r['pool_gate']:<8} "
          f"ates={r['gate_orphan_fired'] or '-':<3} {r['parent_verdict']:<11} {r['parent_name'][:28]:<30} {r['orphan_tokens']}")
d = {r['query']: r for r in rows}
assert all(r['prompt_variant'] == 'v4' for r in rows), 'v4 devrede degil'
assert d['AFAD']['gate_orphan_fired'] == '0', 'kapi AFAD"ta yanlis ateşledi'
print('\nDuman testi tamam.')


## 10) İşçi hız testi

1 / 2 / 4 / 8 işçi, 30 satırda. Ölçüm 2026-08-11'de yerelde 4'te ~2,2x demişti,
8'de kazanç geriliyordu — Colab GPU'sunda doğrulanacak. **Karar bu tablodan verilir**;
aşağıdaki tam koşu hücresinde `WORKERS`'ı buna göre ayarla (varsayılan 4).


In [ ]:
import os, csv, time
os.chdir(REPO)
sonuc = []
for w in (1, 2, 4, 8):
    out = f'/content/tmp/w{w}.csv'
    if os.path.exists(out): os.remove(out)
    t0 = time.time()
    !python3 -m institution_resolver_v3.cli.main inventory-batch '{GIRDI}' --out {out} --limit 30 --workers {w} > /content/tmp/w{w}.log 2>&1
    dt = time.time() - t0
    n = sum(1 for _ in csv.DictReader(open(out, encoding='utf-8'))) if os.path.exists(out) else 0
    sonuc.append((w, dt, n))
    print(f'workers={w:>2}  {dt:>6.1f} sn  {n} satir  {dt/max(n,1):>5.2f} sn/sorgu')

print()
taban = sonuc[0][1]
for w, dt, n in sonuc:
    gun = 143039 * (dt/max(n,1)) / 86400
    print(f'workers={w:>2}  hizlanma {taban/dt:>4.2f}x  ->  143k icin ~{gun:.1f} gun')


## 11) TAM KOŞU

Her `CHUNK` satırda Drive'a yedekler. Kesinti olursa bu hücreyi tekrar çalıştır —
`--resume` çıktıdaki `query` metinlerini okuyup tamamlananları atlar.


In [ ]:
import os, shutil, time, csv
os.chdir(REPO)

WORKERS = 4                      # 10. hücredeki tabloya gore ayarla
CHUNK   = 500                    # her bu kadar YENI satirda Drive yedegi
YEREL   = '/content/inventory_v4ec.csv'
DRIVE   = f'{OUTPUT}/inventory_v4ec.csv'

if os.path.exists(DRIVE) and not os.path.exists(YEREL):
    shutil.copy(DRIVE, YEREL)    # onceki oturumdan devam

def _satir(p):
    if not os.path.exists(p): return 0
    csv.field_size_limit(10_000_000)
    with open(p, newline='', encoding='utf-8') as f:
        return sum(1 for _ in csv.DictReader(f))

HEDEF = 143039
while True:
    onceki = _satir(YEREL)
    if onceki >= HEDEF:
        print('TAMAMLANDI:', onceki); break
    !python3 -m institution_resolver_v3.cli.main inventory-batch '{GIRDI}' \
        --out {YEREL} --workers {WORKERS} --limit {CHUNK} --resume
    simdi = _satir(YEREL)
    shutil.copy(YEREL, DRIVE)
    print(f'  {simdi}/{HEDEF} satir  (+{simdi-onceki})  -> Drive yedeklendi  {time.strftime("%H:%M")}')
    if simdi == onceki:
        print('ILERLEME YOK - duruluyor'); break


## 12) Özet


In [ ]:
import csv, collections
csv.field_size_limit(10_000_000)
rows = list(csv.DictReader(open(YEREL, encoding='utf-8')))
print(f'{len(rows)} satir\n')
print('parent karari:', dict(collections.Counter(r['parent_verdict'] or '(bos)' for r in rows)))
print('karar veren  :', dict(collections.Counter(r['parent_decided_by'] or '(yok)' for r in rows)))
ates = sum(1 for r in rows if r['gate_orphan_fired'] == '1')
print(f'kapi ateşleyen: {ates} ({100*ates/max(len(rows),1):.1f}%)')
print('durum        :', dict(collections.Counter(r['status'] for r in rows)))
hata = collections.Counter(r['error'][:60] for r in rows if r['status'] != 'ok')
for k, v in hata.most_common(5): print(f'  {v:>5}  {k}')

import shutil; shutil.copy(YEREL, DRIVE)
print('\nSon hali Drive"a yedeklendi:', DRIVE)
